# Tech Challenge - Fase 1

## Desafio

Um grande hospital universitário busca implementar um sistema inteligente de suporte ao diagnóstico, capaz de ajudar médicos e equipes clínicas na análise inicial de exames e no processamento de dados médicos.

Com um volume crescente de pacientes e exames, como radiografias, tomografias, ressonâncias e prontuários digitalizados, o hospital precisa de soluções que acelerem a triagem e apoiem as decisões médicas, reduzindo erros e otimizando o tempo dos profissionais.

Nesta primeira fase, o desafio é criar a base do sistema de IA focado em machine learning, permitindo que resultados de exames sejam analisados automaticamente e destacando informações relevantes para o diagnóstico.

## Objetivo

Construir uma solução inicial com foco em IA para processamento de exames médicos e documentos clínicos, aplicando fundamentos essenciais de IA, Machine Learning e Visão Computacional.

## Patologia escolhida para o desafio: Acidente Vascular Cerebral (AVC)

### O que é o AVC

O Acidente Vascular Cerebral (AVC) é uma condição em que o fluxo de sangue para uma parte do cérebro é interrompido. Como as células cerebrais dependem de oxigênio e nutrientes trazidos pelo sangue, essa interrupção pode causar danos neurológicos em poucos minutos.

### Tipos de AVC

1. AVC Isquêmico

- Ocorre quando um vaso sanguíneo é bloqueado por um coágulo.
- É o tipo mais comum.
- Geralmente relacionado a aterosclerose, fibrilação atrial e fatores de risco cardiovasculares.

2. AVC Hemorrágico
- Acontece quando um vaso sanguíneo se rompe, causando sangramento no cérebro.
- Pode ser causado por hipertensão não controlada, aneurismas ou malformações vasculares.

### Base de Dados - NHANES (https://www.cdc.gov/nchs/nhanes)

O NHANES (National Health and Nutrition Examination Survey) é uma base de dados pública conduzida pelo CDC (Centers for Disease Control and Prevention) dos Estados Unidos, voltada à avaliação do estado de saúde e nutricional da população norte-americana.

De forma breve:

- Natureza: estudo observacional, transversal, com amostragem probabilística representativa da população dos EUA.

- Periodicidade: realizado continuamente, organizado em ciclos bienais (ex.: 2017–2018 https://wwwn.cdc.gov/Nchs/Nhanes/continuousnhanes/default.aspx?BeginYear=2017).

- Conteúdo: combina

    - Questionários (doenças autorreferidas, estilo de vida, tabagismo, diabetes, histórico de AVC),

    - Exames físicos (pressão arterial, IMC, medidas corporais),

    - Exames laboratoriais (glicemia, hemoglobina glicada, colesterol, triglicerídeos, marcadores inflamatórios).

- Formato dos dados: arquivos modulares (.XPT), integrados por um identificador único do participante (SEQN).

- Aspecto metodológico importante: inclui pesos amostrais e variáveis de desenho complexo, permitindo inferência populacional.

No contexto acadêmico, o NHANES é amplamente utilizado para modelagem de risco cardiovascular e de AVC, epidemiologia, análise de fatores de risco e aplicações de machine learning em saúde, sendo valorizado pela qualidade dos dados, transparência metodológica e acesso aberto.

**Importando a base de dados**

In [26]:
import pandas as pd  # importa a biblioteca pandas

modules = {  # dicionário que mapeia nomes lógicos para arquivos XPT
    "demo": "DEMO_J.XPT",  # arquivo DEMO (dados demográficos): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.htm
    "bpx": "BPX_J.XPT",     # arquivo BPX (medidas de pressão arterial): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BPX_J.htm
    "bpq": "BPQ_J.XPT", # arquivo BPQ (pressão arterial e colesterol): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BPQ_J.htm
    "ocq": "OCQ_J.XPT",     # arquivo OCQ (questionário de ocupação): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/OCQ_J.htm
    "ghb": "GHB_J.XPT",     # arquivo GHB (hemoglobina glicada): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/GHB_J.htm
    "bmx": "BMX_J.XPT",     # arquivo BMX (medidas corporais): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BMX_J.htm
    "smq": "SMQ_J.XPT",     # arquivo SMQ (questionário de tabagismo): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/SMQ_J.htm
    "mcq": "MCQ_J.XPT"     # arquivo de questionário médico (histórico de doenças): https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/MCQ_J.htm
}

dfs = {}  # dicionário vazio para armazenar os DataFrames lidos de cada arquivo
base_url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/"  # URL base para os arquivos NHANES 2017-2018

for name, file in modules.items():  # itera sobre (nome, arquivo) do dicionário modules
    dfs[name] = pd.read_sas(base_url + file)  # lê o arquivo XPT remoto e guarda em dfs[name]

# inicia com o DataFrame 'demo' e faz merges left sucessivos com os outros DataFrames usando 'SEQN' como chave
df = dfs["demo"] \
    .merge(dfs["bpx"], on="SEQN", how="left") \
    .merge(dfs["bpq"], on="SEQN", how="left") \
    .merge(dfs["ocq"], on="SEQN", how="left") \
    .merge(dfs["ghb"], on="SEQN", how="left") \
    .merge(dfs["bmx"], on="SEQN", how="left") \
    .merge(dfs["smq"], on="SEQN", how="left") \
    .merge(dfs["mcq"], on="SEQN", how="left")


In [27]:
df.shape

(9254, 217)

In [28]:
# manter somente as colunas solicitadas (se existirem) e avisar se alguma estiver ausente
cols_to_keep = [
    "SEQN", # Respondent sequence number
    "RIAGENDR", # Gender
    "RIDAGEYR", # Age in years at screening 
    "BPQ020", # Ever told you had high blood pressure
    "MCQ160B", # Ever told had congestive heart failure
    "MCQ160C", # Age when told you had heart failure
    "MCQ160D", # Age when told had coronary heart disease
    "MCQ160E", # Age when told you had angina pectoris
    "DMDMARTL", # Marital status
    "OCQ260", # Description of job/work situation
    "LBXGH", # Glycohemoglobin
    "BMXBMI", #  Body Mass Index (BMI)
    "SMQ020", # Smoked at least 100 cigarettes in life
    "MCQ160F" #  Ever told you had a stroke
]

missing = [c for c in cols_to_keep if c not in df.columns]
if missing:
    print("Aviso: colunas ausentes e que não foram selecionadas:", missing)

df = df[[c for c in cols_to_keep if c in df.columns]].copy()
df.shape

(9254, 14)

**Exploração de dados**

In [29]:
df.head(5)

,SEQN,RIAGENDR,RIDAGEYR,BPQ020,MCQ160B,MCQ160C,MCQ160D,MCQ160E,DMDMARTL,OCQ260,LBXGH,BMXBMI,SMQ020,MCQ160F
0,93703.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.5,NaN,NaN
1,93704.0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.7,NaN,NaN
2,93705.0,2.0,66.0,1.0,2.0,2.0,2.0,2.0,3.0,1.0,6.2,31.7,1.0,2.0
3,93706.0,1.0,18.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,5.2,21.5,2.0,NaN
4,93707.0,1.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.6,18.1,NaN,NaN


In [25]:
df.info

<bound method DataFrame.info of           SEQN  SDDSRVYR  RIDSTATR  RIAGENDR  RIDAGEYR  RIDAGEMN  RIDRETH1  \
0      93703.0      10.0       2.0       2.0       2.0       NaN       5.0   
1      93704.0      10.0       2.0       1.0       2.0       NaN       3.0   
2      93705.0      10.0       2.0       2.0      66.0       NaN       4.0   
3      93706.0      10.0       2.0       1.0      18.0       NaN       5.0   
4      93707.0      10.0       2.0       1.0      13.0       NaN       5.0   
...        ...       ...       ...       ...       ...       ...       ...   
9249  102952.0      10.0       2.0       2.0      70.0       NaN       5.0   
9250  102953.0      10.0       2.0       1.0      42.0       NaN       1.0   
9251  102954.0      10.0       2.0       2.0      41.0       NaN       4.0   
9252  102955.0      10.0       2.0       2.0      14.0       NaN       4.0   
9253  102956.0      10.0       2.0       1.0      38.0       NaN       3.0   

      RIDRETH3  RIDEXMON  RIDEX

## Referências

- https://pandascouple.medium.com/projeto-machine-learning-previs%C3%A3o-de-avc-f4b7dce11929
- https://jornal.usp.br/radio-usp/uso-de-ia-e-analise-de-dados-na-prevencao-de-avc-e-ataque-isquemico-transitorio/
- https://www-nature-com.translate.goog/articles/s41598-024-61665-4?error=cookies_not_supported&code=1d85f26e-6ade-4ec5-9132-5511aa615597&_x_tr_sl=en&_x_tr_tl=pt&_x_tr_hl=pt&_x_tr_pto=tc
- https://jhi.sbis.org.br/index.php/jhi-sbis/article/view/980
